# CAMELYON16 → embedding H-optimus-0 su Colab

Esegue `td task=stream`: scarica ogni vetrino sul disco della VM, calcola maschera tessuto, QC e griglia di tile, estrae gli embedding sulla GPU, li salva su **Google Drive** e cancella il vetrino.

- **Riprendibile.** Colab gratuito si disconnette dopo al massimo ~12 h: basta rieseguire tutte le celle, i vetrini già su Drive vengono saltati senza riscaricarli.
- **Spazio.** Circa 120 MB per vetrino in float16 (40.000 tile × 1536 dimensioni): controlla lo spazio libero su Drive prima di lanciare tutti i 399 vetrini.
- **Solo dati pubblici.** Nessuna immagine clinica va caricata su Colab o su un Drive personale.
- Runtime → Cambia tipo di runtime → **GPU T4** prima di iniziare.

In [ ]:
# 1. Hardware: GPU, CPU e disco locale (i vetrini passano di qui)
!nvidia-smi -L
!nproc
!df -h /content | tail -1

In [ ]:
# 2. Google Drive: gli embedding finiscono qui, così sopravvivono alle disconnessioni
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tumordetector"
!mkdir -p {OUT}

In [ ]:
# 3. Codice (se il repository è privato, clonalo con un token o rendilo pubblico)
REPO_URL = "https://github.com/masnarivan/TumorDetector.git"
REV = "main"    # meglio un commit preciso: finisce nei metadati di ogni vetrino
!git clone -q {REPO_URL} /content/TumorDetector 2>/dev/null || git -C /content/TumorDetector fetch -q
!git -C /content/TumorDetector checkout -q {REV}
!pip install -q -e /content/TumorDetector

In [ ]:
# 4. Pesi di H-optimus-0 (modello ad accesso controllato: accetta prima i termini sulla pagina
#    https://huggingface.co/bioptimus/H-optimus-0, poi inserisci qui il tuo token di lettura)
from huggingface_hub import notebook_login, snapshot_download
notebook_login()

In [ ]:
snapshot_download("bioptimus/H-optimus-0")

In [ ]:
# 5. Prova su 2 vetrini: misura il tempo per vetrino prima di lanciare tutto
%cd /content/TumorDetector
!td task=stream backbone=h_optimus_0 data.raw_dir=/content/raw paths.out={OUT} \
    mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db "data.slides=[tumor_031,normal_108]"

In [ ]:
# 6. Estrazione completa (data.slides=null = tutti i vetrini del manifest)
!td task=stream backbone=h_optimus_0 data.raw_dir=/content/raw paths.out={OUT} \
    mlflow.tracking_uri=sqlite:///{OUT}/mlflow.db data.slides=null

In [ ]:
# 7. Stato (eseguibile in qualsiasi momento, anche da una nuova sessione)
import json, pathlib
import pandas as pd
root = pathlib.Path(OUT, "embeddings", "h_optimus_0")
print(json.loads((root / "collection.json").read_text()))
print(len(list(root.glob("*.npy"))), "vetrini con embedding")
log = pd.read_csv(root / "extract_log.csv")   # ultima esecuzione
done = log[log.status == "embedded"]
print(f"ultima esecuzione: {done.n_tiles.sum():,.0f} tile, {done.n_tiles.sum() / max(done.t_embed_s.sum(), 1):.0f} tile/s")
log[log.status.isin(["error"]) | log.status.str.startswith("referred")]

Alla fine: scarica la cartella `tumordetector/embeddings/h_optimus_0` da Drive in `data/embeddings/h_optimus_0` sul Mac, imposta `backbone: h_optimus_0` in `dvc.yaml` ed esegui `td task=index backbone=h_optimus_0`.